# 戦略1: バリュエーション水準と資本効率性指標の有効性モデル

**作成日**: 2026-02-21

**戦略**: P/B四分位ごとにROE/ROAの有効性を検証

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import json
import warnings
warnings.filterwarnings('ignore')

print("ライブラリインポート完了")

ライブラリインポート完了


## 1. データ読み込み・前処理

In [2]:
PROJECT_ROOT = Path(r'C:\Users\yongr\claude project\workspace')

# 価格データ
print("価格データ読み込み中...")
df_price = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/prices/daily_quotes_all.parquet')
df_price['date'] = pd.to_datetime(df_price['date'])
df_price = df_price[df_price['date'] >= '2017-01-01'].copy()
print(f"価格データ: {len(df_price):,} 行")

# 財務データ
print("財務データ読み込み中...")
df_fin = pd.read_parquet(PROJECT_ROOT / 'data/curated/jquants/financials/statements_all.parquet')
df_fin['disclosed_date'] = pd.to_datetime(df_fin['disclosed_date'])
df_fin = df_fin[df_fin['disclosed_date'] >= '2017-01-01'].copy()

# 年次決算のみに限定
if 'fiscal_quarter' in df_fin.columns:
    before_count = len(df_fin)
    df_fin = df_fin[df_fin['fiscal_quarter'] == 'FY'].copy()
    print(f"年次決算フィルタ: {before_count:,} → {len(df_fin):,} 行")

# 必須カラムのみ抽出
df_fin = df_fin[['disclosed_date', 'code', 'equity', 'net_profit', 'total_assets', 'bps']].copy()

# ROE, ROA計算
df_fin['roe'] = (df_fin['net_profit'] / df_fin['equity']) * 100
df_fin['roa'] = (df_fin['net_profit'] / df_fin['total_assets']) * 100

# 異常値除外
df_fin = df_fin[
    (df_fin['roe'] > -100) & (df_fin['roe'] < 100) &
    (df_fin['roa'] > -100) & (df_fin['roa'] < 100) &
    (df_fin['bps'] > 0) & (df_fin['equity'] > 0) & (df_fin['total_assets'] > 0)
].copy()

print(f"財務データ（年次のみ、異常値除外後）: {len(df_fin):,} 行")
print("前処理完了")

価格データ読み込み中...


価格データ: 9,148,457 行
財務データ読み込み中...


年次決算フィルタ: 171,943 → 60,649 行
財務データ（年次のみ、異常値除外後）: 37,203 行
前処理完了


## 2. 価格データのピボット化

In [3]:
print("価格データをピボット化中...")
df_price_pivot = df_price.pivot(index='date', columns='code', values='adjusted_close')
print(f"ピボットテーブル: {df_price_pivot.shape[0]} 日 × {df_price_pivot.shape[1]} 銘柄")
print("ピボット化完了")

価格データをピボット化中...


ピボットテーブル: 2212 日 × 5226 銘柄
ピボット化完了


## 3. リバランス日生成（月次）

In [4]:
# 月次リバランス日（月末営業日）
trading_days = pd.DataFrame({'date': df_price_pivot.index})
trading_days['year'] = trading_days['date'].dt.year
trading_days['month'] = trading_days['date'].dt.month
rebalance_dates = trading_days.groupby(['year', 'month'])['date'].max().values
rebalance_dates = pd.Series(rebalance_dates).sort_values().reset_index(drop=True)

print(f"リバランス日数: {len(rebalance_dates)}")
print(f"期間: {rebalance_dates.iloc[0].date()} ~ {rebalance_dates.iloc[-1].date()}")

リバランス日数: 109
期間: 2017-01-31 ~ 2026-01-22


## 4. 財務データの事前処理

In [5]:
print("財務データの事前処理中...")

fin_by_date = {}

for i, rdate in enumerate(rebalance_dates):
    if i % 10 == 0:
        print(f"  進捗: {i}/{len(rebalance_dates)}")
    
    # その日までに開示された財務データ
    available = df_fin[df_fin['disclosed_date'] <= rdate].copy()
    
    # 各銘柄の最新データ
    latest = available.sort_values('disclosed_date').groupby('code').tail(1)
    latest = latest.set_index('code')[['bps', 'roe', 'roa']]
    
    fin_by_date[rdate] = latest

print("財務データ事前処理完了")

財務データの事前処理中...
  進捗: 0/109
  進捗: 10/109


  進捗: 20/109
  進捗: 30/109


  進捗: 40/109
  進捗: 50/109


  進捗: 60/109
  進捗: 70/109


  進捗: 80/109


  進捗: 90/109


  進捗: 100/109


財務データ事前処理完了


## 5. スクリーニング関数（PBR四分位 × ROE/ROA）

In [6]:
def screen_stocks_by_quartile(rebalance_date, prices_pivot, fin_data, n_stocks=20):
    """
    PBR四分位ごとに、ROE/ROAでポートフォリオを構築
    
    Returns:
        dict: {
            'Q1_ROE': [codes],
            'Q1_ROA': [codes],
            'Q2_ROE': [codes],
            'Q2_ROA': [codes],
            'Q3_ROE': [codes],
            'Q3_ROA': [codes],
            'Q4_ROE': [codes],
            'Q4_ROA': [codes]
        }
    """
    # その日の価格
    if rebalance_date not in prices_pivot.index:
        return {}
    
    prices = prices_pivot.loc[rebalance_date].dropna()
    
    # 財務データ
    if rebalance_date not in fin_data:
        return {}
    
    fin = fin_data[rebalance_date]
    
    # マージ
    merged = pd.DataFrame({
        'adjusted_close': prices,
        'bps': fin['bps'],
        'roe': fin['roe'],
        'roa': fin['roa']
    }).dropna()
    
    if len(merged) < n_stocks * 4:  # 最低でも4つの四分位に分割可能
        return {}
    
    # PBR計算
    merged['pbr'] = merged['adjusted_close'] / merged['bps']
    
    # 異常値除外
    merged = merged[(merged['pbr'] > 0.01) & (merged['pbr'] < 50)]
    
    if len(merged) < n_stocks * 4:
        return {}
    
    # PBR四分位を計算
    merged['pbr_quartile'] = pd.qcut(merged['pbr'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
    
    # 各四分位でROE/ROAの上位25%を選択
    portfolios = {}
    
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        q_data = merged[merged['pbr_quartile'] == q]
        
        if len(q_data) < n_stocks:
            # データが少ない場合はスキップ
            portfolios[f'{q}_ROE'] = pd.DataFrame()
            portfolios[f'{q}_ROA'] = pd.DataFrame()
            continue
        
        # ROE上位
        roe_top = q_data.nlargest(n_stocks, 'roe')
        portfolios[f'{q}_ROE'] = roe_top.reset_index()
        
        # ROA上位
        roa_top = q_data.nlargest(n_stocks, 'roa')
        portfolios[f'{q}_ROA'] = roa_top.reset_index()
    
    return portfolios

# テスト
test_date = rebalance_dates.iloc[10]
test_result = screen_stocks_by_quartile(test_date, df_price_pivot, fin_by_date)
print(f"テスト結果（{test_date.date()}）:")
for key, df in test_result.items():
    print(f"  {key}: {len(df)} 銘柄")

テスト結果（2017-11-30）:
  Q1_ROE: 20 銘柄
  Q1_ROA: 20 銘柄
  Q2_ROE: 20 銘柄
  Q2_ROA: 20 銘柄
  Q3_ROE: 20 銘柄
  Q3_ROA: 20 銘柄
  Q4_ROE: 20 銘柄
  Q4_ROA: 20 銘柄


## 6. バックテストループ（8ポートフォリオ）

In [7]:
# パラメータ
INITIAL_CASH = 10_000_000
N_STOCKS = 20
TAX_RATE = 0.20315
UNIT = 100

# 8つのポートフォリオを管理
portfolio_names = ['Q1_ROE', 'Q1_ROA', 'Q2_ROE', 'Q2_ROA', 
                   'Q3_ROE', 'Q3_ROA', 'Q4_ROE', 'Q4_ROA']

# 各ポートフォリオの初期化
portfolios_state = {name: {
    'cash': INITIAL_CASH,
    'portfolio': {},
    'annual_realized_pnl': 0,
    'current_year': None,
    'results': []
} for name in portfolio_names}

print("バックテスト開始（8ポートフォリオ）...")
print(f"初期資本（各ポートフォリオ）: {INITIAL_CASH:,}円")
print(f"リバランス回数: {len(rebalance_dates)}")
print()

for i, rebalance_date in enumerate(rebalance_dates):
    if i % 12 == 0:
        print(f"進捗: {i}/{len(rebalance_dates)} ({i/len(rebalance_dates)*100:.1f}%) - {rebalance_date.date()}")
    
    # 銘柄選定（8ポートフォリオ分）
    selected_portfolios = screen_stocks_by_quartile(rebalance_date, df_price_pivot, fin_by_date, N_STOCKS)
    
    # 各ポートフォリオで処理
    for pf_name in portfolio_names:
        state = portfolios_state[pf_name]
        
        # 年の切り替わり
        if state['current_year'] != rebalance_date.year:
            if state['current_year'] is not None and state['annual_realized_pnl'] > 0:
                tax = state['annual_realized_pnl'] * TAX_RATE
                state['cash'] -= tax
            state['annual_realized_pnl'] = 0
            state['current_year'] = rebalance_date.year
        
        # 既存ポートフォリオ売却
        sell_value = 0
        if rebalance_date in df_price_pivot.index:
            for code, position in state['portfolio'].items():
                if code in df_price_pivot.columns:
                    sell_price = df_price_pivot.loc[rebalance_date, code]
                    if pd.notna(sell_price):
                        sell_amount = position['shares'] * sell_price
                        sell_value += sell_amount
                        pnl = (sell_price - position['buy_price']) * position['shares']
                        state['annual_realized_pnl'] += pnl
        
        state['cash'] += sell_value
        state['portfolio'] = {}
        
        # 銘柄選定結果を取得
        selected = selected_portfolios.get(pf_name, pd.DataFrame())
        
        if len(selected) == 0:
            state['results'].append({
                'date': rebalance_date,
                'cash': state['cash'],
                'n_stocks': 0,
                'invested': 0,
                'annual_pnl': state['annual_realized_pnl']
            })
            continue
        
        # 購入
        target_per_stock = state['cash'] / len(selected)
        total_invested = 0
        
        for _, row in selected.iterrows():
            code = row['code']
            price = row['adjusted_close']
            shares = int(target_per_stock / (price * UNIT)) * UNIT
            
            if shares > 0:
                invest_amount = shares * price
                total_invested += invest_amount
                state['portfolio'][code] = {'shares': shares, 'buy_price': price}
        
        state['cash'] -= total_invested
        
        state['results'].append({
            'date': rebalance_date,
            'cash': state['cash'],
            'n_stocks': len(state['portfolio']),
            'invested': total_invested,
            'annual_pnl': state['annual_realized_pnl']
        })

# 最終税金と時価評価
final_date = df_price_pivot.index.max()
print(f"\n最終営業日: {final_date.date()}")

for pf_name in portfolio_names:
    state = portfolios_state[pf_name]
    
    # 最終税金
    if state['annual_realized_pnl'] > 0:
        tax = state['annual_realized_pnl'] * TAX_RATE
        state['cash'] -= tax
        state['results'][-1]['cash'] = state['cash']
    
    # 最終日時点での保有株式を時価評価
    final_portfolio_value = 0
    if len(state['portfolio']) > 0:
        for code, position in state['portfolio'].items():
            if code in df_price_pivot.columns:
                final_price = df_price_pivot.loc[final_date, code]
                if pd.notna(final_price):
                    final_portfolio_value += position['shares'] * final_price
        state['results'][-1]['invested'] = final_portfolio_value

print("\nバックテスト完了")

バックテスト開始（8ポートフォリオ）...
初期資本（各ポートフォリオ）: 10,000,000円
リバランス回数: 109

進捗: 0/109 (0.0%) - 2017-01-31


進捗: 12/109 (11.0%) - 2018-01-31


進捗: 24/109 (22.0%) - 2019-01-31


進捗: 36/109 (33.0%) - 2020-01-31


進捗: 48/109 (44.0%) - 2021-01-29


進捗: 60/109 (55.0%) - 2022-01-31


進捗: 72/109 (66.1%) - 2023-01-31


進捗: 84/109 (77.1%) - 2024-01-31


進捗: 96/109 (88.1%) - 2025-01-31


進捗: 108/109 (99.1%) - 2026-01-22

最終営業日: 2026-01-22

バックテスト完了


## 7. パフォーマンス分析

In [8]:
# 各ポートフォリオの結果をDataFrameに変換
portfolios_df = {}

for pf_name in portfolio_names:
    state = portfolios_state[pf_name]
    df = pd.DataFrame(state['results'])
    df['total_value'] = df['cash'] + df['invested']
    df['return'] = df['total_value'].pct_change()
    df['cumulative_return'] = (1 + df['return']).cumprod() - 1
    df['peak'] = df['total_value'].cummax()
    df['drawdown'] = (df['total_value'] - df['peak']) / df['peak']
    portfolios_df[pf_name] = df

print("各ポートフォリオのDataFrame作成完了")

各ポートフォリオのDataFrame作成完了


In [9]:
# パフォーマンスサマリ計算
summary_data = []

for pf_name in portfolio_names:
    df = portfolios_df[pf_name]
    
    # 期間
    years = (df['date'].iloc[-1] - df['date'].iloc[0]).days / 365.25
    
    # リターン
    total_return = df['cumulative_return'].iloc[-1]
    annual_return = (1 + total_return) ** (1 / years) - 1
    
    # ボラティリティ
    volatility = df['return'].std() * np.sqrt(12)  # 月次なので√12
    
    # MDD
    mdd = df['drawdown'].min()
    
    # シャープレシオ
    sharpe = df['return'].mean() / df['return'].std() * np.sqrt(12) if df['return'].std() > 0 else 0
    
    # カルマー比
    calmar = annual_return / abs(mdd) if mdd != 0 else 0
    
    # 最終資産
    final_value = df['total_value'].iloc[-1]
    
    summary_data.append({
        'ポートフォリオ': pf_name,
        '最終資産（円）': f"{final_value:,.0f}",
        '総リターン（%）': f"{total_return*100:.2f}",
        '年率リターン（%）': f"{annual_return*100:.2f}",
        '年率ボラティリティ（%）': f"{volatility*100:.2f}",
        '最大DD（%）': f"{mdd*100:.2f}",
        'シャープレシオ': f"{sharpe:.2f}",
        'カルマー比': f"{calmar:.2f}"
    })

df_summary = pd.DataFrame(summary_data)

print("="*100)
print("パフォーマンスサマリ（全8ポートフォリオ）")
print("="*100)
print(f"期間: {df['date'].iloc[0].date()} ~ {df['date'].iloc[-1].date()}")
print(f"初期資本: {INITIAL_CASH:,}円")
print()
display(df_summary)
print("="*100)

パフォーマンスサマリ（全8ポートフォリオ）
期間: 2017-01-31 ~ 2026-01-22
初期資本: 10,000,000円



,ポートフォリオ,最終資産（円）,総リターン（%）,年率リターン（%）,年率ボラティリティ（%）,最大DD（%）,シャープレシオ,カルマー比
0,Q1_ROE,"40,073,018",300.73,16.73,21.98,-35.45,0.81,0.47
1,Q1_ROA,"34,080,491",240.80,14.64,18.70,-29.02,0.83,0.50
2,Q2_ROE,"31,172,894",211.73,13.51,19.96,-36.61,0.74,0.37
3,Q2_ROA,"33,327,896",233.28,14.35,19.67,-33.47,0.78,0.43
4,Q3_ROE,"12,156,033",21.56,2.20,19.51,-33.12,0.21,0.07
5,Q3_ROA,"11,956,911",19.57,2.01,17.46,-29.41,0.20,0.07
6,Q4_ROE,"12,116,528",21.17,2.16,18.71,-34.03,0.21,0.06
7,Q4_ROA,"9,072,507",-9.27,-1.08,17.41,-39.40,0.03,-0.03


## 8. 比較分析: PBR四分位ごとのROE vs ROA

In [10]:
# 各四分位でROE vs ROAを比較
print("="*100)
print("PBR四分位ごとの比較: ROE vs ROA")
print("="*100)

for q in ['Q1', 'Q2', 'Q3', 'Q4']:
    roe_name = f"{q}_ROE"
    roa_name = f"{q}_ROA"
    
    roe_final = portfolios_df[roe_name]['total_value'].iloc[-1]
    roa_final = portfolios_df[roa_name]['total_value'].iloc[-1]
    
    roe_return = portfolios_df[roe_name]['cumulative_return'].iloc[-1] * 100
    roa_return = portfolios_df[roa_name]['cumulative_return'].iloc[-1] * 100
    
    print(f"\n{q}（PBR {'最低' if q=='Q1' else '低-中' if q=='Q2' else '中-高' if q=='Q3' else '最高'}四分位）:")
    print(f"  ROE戦略: 最終資産={roe_final:,.0f}円、リターン={roe_return:.2f}%")
    print(f"  ROA戦略: 最終資産={roa_final:,.0f}円、リターン={roa_return:.2f}%")
    
    if roe_final > roa_final:
        diff = (roe_final / roa_final - 1) * 100
        print(f"  → ROEが優位（+{diff:.2f}%）")
    else:
        diff = (roa_final / roe_final - 1) * 100
        print(f"  → ROAが優位（+{diff:.2f}%）")

print("\n" + "="*100)

PBR四分位ごとの比較: ROE vs ROA

Q1（PBR 最低四分位）:
  ROE戦略: 最終資産=40,073,018円、リターン=300.73%
  ROA戦略: 最終資産=34,080,491円、リターン=240.80%
  → ROEが優位（+17.58%）

Q2（PBR 低-中四分位）:
  ROE戦略: 最終資産=31,172,894円、リターン=211.73%
  ROA戦略: 最終資産=33,327,896円、リターン=233.28%
  → ROAが優位（+6.91%）

Q3（PBR 中-高四分位）:
  ROE戦略: 最終資産=12,156,033円、リターン=21.56%
  ROA戦略: 最終資産=11,956,911円、リターン=19.57%
  → ROEが優位（+1.67%）

Q4（PBR 最高四分位）:
  ROE戦略: 最終資産=12,116,528円、リターン=21.17%
  ROA戦略: 最終資産=9,072,507円、リターン=-9.27%
  → ROEが優位（+33.55%）



## 9. ベストポートフォリオの特定

In [11]:
# 最終資産が最大のポートフォリオ
best_portfolio = None
best_value = 0

for pf_name in portfolio_names:
    final_value = portfolios_df[pf_name]['total_value'].iloc[-1]
    if final_value > best_value:
        best_value = final_value
        best_portfolio = pf_name

print("="*100)
print("ベストパフォーマンスポートフォリオ")
print("="*100)
print(f"ポートフォリオ: {best_portfolio}")
print(f"最終資産: {best_value:,.0f}円")
print(f"総リターン: {portfolios_df[best_portfolio]['cumulative_return'].iloc[-1]*100:.2f}%")
print(f"年率リターン: {((1 + portfolios_df[best_portfolio]['cumulative_return'].iloc[-1]) ** (1 / years) - 1)*100:.2f}%")
print(f"最大DD: {portfolios_df[best_portfolio]['drawdown'].min()*100:.2f}%")
print("="*100)

ベストパフォーマンスポートフォリオ
ポートフォリオ: Q1_ROE
最終資産: 40,073,018円
総リターン: 300.73%
年率リターン: 16.73%
最大DD: -35.45%


## 10. 結果の保存

In [12]:
output_dir = PROJECT_ROOT / 'analyses/20260221_0915_quants_model_valuation_efficiency'

# 1. 日次パフォーマンスをCSVに保存（全8ポートフォリオを横結合）
df_combined = portfolios_df[portfolio_names[0]][['date']].copy()

for pf_name in portfolio_names:
    df = portfolios_df[pf_name]
    df_combined[f'{pf_name}_value'] = df['total_value'].values
    df_combined[f'{pf_name}_return'] = df['cumulative_return'].values

output_csv = output_dir / 'backtest_results.csv'
df_combined.to_csv(output_csv, index=False, encoding='utf-8-sig')
print(f"結果を保存: {output_csv}")

# 2. 評価指標をJSONに保存
metrics = {}
for pf_name in portfolio_names:
    df = portfolios_df[pf_name]
    years = (df['date'].iloc[-1] - df['date'].iloc[0]).days / 365.25
    total_return = df['cumulative_return'].iloc[-1]
    annual_return = (1 + total_return) ** (1 / years) - 1
    volatility = df['return'].std() * np.sqrt(12)
    mdd = df['drawdown'].min()
    sharpe = df['return'].mean() / df['return'].std() * np.sqrt(12) if df['return'].std() > 0 else 0
    calmar = annual_return / abs(mdd) if mdd != 0 else 0
    
    metrics[pf_name] = {
        'final_value': float(df['total_value'].iloc[-1]),
        'total_return': float(total_return),
        'annual_return': float(annual_return),
        'volatility': float(volatility),
        'max_drawdown': float(mdd),
        'sharpe_ratio': float(sharpe),
        'calmar_ratio': float(calmar)
    }

output_json = output_dir / 'backtest_metrics.json'
with open(output_json, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)
print(f"評価指標を保存: {output_json}")

# 3. サマリをテキストファイルに保存
output_txt = output_dir / 'performance_summary.txt'
with open(output_txt, 'w', encoding='utf-8') as f:
    f.write("戦略1: バリュエーション水準と資本効率性指標の有効性モデル\n")
    f.write("="*100 + "\n")
    f.write(f"期間: {df['date'].iloc[0].date()} ~ {df['date'].iloc[-1].date()}\n")
    f.write(f"初期資本: {INITIAL_CASH:,}円\n")
    f.write(f"リバランス頻度: 月次\n")
    f.write(f"保有銘柄数: {N_STOCKS}銘柄\n")
    f.write("\n")
    f.write("パフォーマンスサマリ（全8ポートフォリオ）\n")
    f.write("-"*100 + "\n")
    f.write(df_summary.to_string(index=False))
    f.write("\n\n")
    f.write("ベストポートフォリオ\n")
    f.write("-"*100 + "\n")
    f.write(f"ポートフォリオ: {best_portfolio}\n")
    f.write(f"最終資産: {best_value:,.0f}円\n")
    f.write(f"総リターン: {portfolios_df[best_portfolio]['cumulative_return'].iloc[-1]*100:.2f}%\n")
    f.write("\n")
    f.write("PBR四分位ごとの比較\n")
    f.write("-"*100 + "\n")
    for q in ['Q1', 'Q2', 'Q3', 'Q4']:
        roe_name = f"{q}_ROE"
        roa_name = f"{q}_ROA"
        roe_final = portfolios_df[roe_name]['total_value'].iloc[-1]
        roa_final = portfolios_df[roa_name]['total_value'].iloc[-1]
        winner = "ROE" if roe_final > roa_final else "ROA"
        diff = abs(roe_final - roa_final) / min(roe_final, roa_final) * 100
        f.write(f"{q}: {winner}が優位（差: {diff:.2f}%）\n")

print(f"サマリを保存: {output_txt}")
print("\n完了！")

結果を保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0915_quants_model_valuation_efficiency\backtest_results.csv
評価指標を保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0915_quants_model_valuation_efficiency\backtest_metrics.json
サマリを保存: C:\Users\yongr\claude project\workspace\analyses\20260221_0915_quants_model_valuation_efficiency\performance_summary.txt

完了！
